# First Visualizations of Train Delays

In this notebook, we create the first visualizations based on the processed MMTIS data.

The goal is to explore delay patterns by:
- hour of day
- weekday
- station pairs

These visualizations will later become part of the final dashboard.

## 1. Import libraries

In [17]:
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

## 2. Load processed summary tables

In [18]:
from pathlib import Path

project_root = Path.cwd().parent

processed_path = project_root / "data" / "processed"

hour_summary = pd.read_csv(
    processed_path / "mmtis_hour_summary.csv"
)

weekday_summary = pd.read_csv(
    processed_path / "mmtis_weekday_summary.csv"
)

station_pair_summary = pd.read_csv(
    processed_path / "mmtis_station_pair_summary.csv"
)

heatmap_summary = pd.read_csv(
    processed_path / "mmtis_heatmap_summary.csv"
)

top_station_pairs = pd.read_csv(
    processed_path / "mmtis_top_station_pairs.csv"
)

In [19]:
hour_summary.head()

,departure_hour,mean_delay,median_delay,number_of_trips
0,0.0,1.063688,-0.066667,3021
1,1.0,1.394905,0.083333,2764
2,2.0,1.482556,0.216667,12529
3,3.0,1.338393,0.300000,40434
4,4.0,1.723193,0.433333,62136


In [20]:
weekday_summary.head()

,weekday,mean_delay,median_delay,number_of_trips
0,Friday,2.208855,0.550000,155824
1,Monday,2.011663,0.516667,153853
2,Saturday,1.345812,0.200000,132538
3,Sunday,1.326890,0.150000,127091
4,Thursday,2.118489,0.566667,153397


In [21]:
top_station_pairs.head()

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
0,PA,MLX,38.968315,26.083333,91
1,MLX,ND G,33.543229,14.991667,32
2,MKI,I W1,23.970000,24.733333,45
3,BPA,VBO,22.715568,15.433333,455
4,PAG,MLX,21.901109,11.666667,1307


## 3. Average delay by hour of day

This visualization shows how train delays change throughout the day.

We use the capped delay metric to reduce the impact of extreme outliers.

In [22]:
fig = px.line(
    hour_summary,
    x="departure_hour",
    y="mean_delay",
    markers=True,
    title="Average Delay by Hour of Day"
)

fig.show()

In [23]:
hour_summary.columns
weekday_summary.columns

Index(['weekday', 'mean_delay', 'median_delay', 'number_of_trips'], dtype='object')

## 4. Average delay by weekday

This chart shows average train delays for each day of the week.

It helps us identify whether some weekdays have systematically higher delays than others.

In [24]:
weekday_summary

,weekday,mean_delay,median_delay,number_of_trips
0,Friday,2.208855,0.550000,155824
1,Monday,2.011663,0.516667,153853
2,Saturday,1.345812,0.200000,132538
3,Sunday,1.326890,0.150000,127091
4,Thursday,2.118489,0.566667,153397
5,Tuesday,2.166582,0.566667,155158
6,Wednesday,2.074099,0.550000,156416


## 5. Average delay by weekday

This chart shows average train delays for each day of the week.

We compare weekdays and weekends to identify possible differences in train punctuality.

In [25]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_summary["weekday"] = pd.Categorical(
    weekday_summary["weekday"],
    categories=weekday_order,
    ordered=True
)

weekday_summary = weekday_summary.sort_values("weekday")

In [26]:
fig = px.bar(
    weekday_summary,
    x="weekday",
    y="mean_delay",
    title="Average Delay by Weekday"
)

fig.show()

### Observation

Train delays are generally higher during weekdays than during weekends.

Friday has the highest average delay, while Saturday and Sunday show the lowest delays.

This may indicate increased traffic and operational pressure during working days.

## 6. Delay heatmap by weekday and hour

This heatmap shows average train delays by:

- day of week
- hour of day

The goal is to identify periods with higher delay levels and possible congestion patterns.

In [27]:
heatmap_data = heatmap_summary.pivot(
    index="weekday",
    columns="departure_hour",
    values="mean_delay"
)

heatmap_data.head()

departure_hour,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,...,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0
weekday,,,,,,,,,,,,,,,,,,,,,
Friday,1.144629,1.570074,1.418238,1.463994,1.930933,2.268178,2.414866,2.313128,2.113925,2.074963,...,2.481975,2.647602,2.453096,2.366320,2.304369,2.248739,1.770452,1.564820,1.332298,1.528809
Monday,3.910176,3.500577,1.696456,1.503336,1.880946,2.290649,2.407457,2.081602,1.922777,1.770067,...,2.300110,2.558335,2.213203,2.269628,2.156004,1.927911,1.680505,1.390417,1.603820,1.933059
Saturday,0.449254,0.571426,1.050694,0.869034,1.019356,1.114148,1.381388,1.500098,1.386659,1.565741,...,1.608534,1.659343,1.524964,1.530757,1.374236,1.434721,1.174938,1.033972,0.976668,1.020234
Sunday,0.553174,1.054768,1.288297,0.803299,0.967207,1.026761,1.083983,1.287959,1.075973,1.233318,...,1.654569,1.802138,1.553772,1.766442,1.445288,1.296676,1.320353,1.218527,1.151065,1.440234
Thursday,2.585386,2.011318,1.573670,1.556642,1.954567,2.369473,2.625829,2.404493,2.100807,1.872569,...,2.322225,2.732033,2.485415,2.316966,2.064068,1.990623,1.660735,1.403687,1.358436,1.442208


In [28]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

heatmap_data = heatmap_data.reindex(weekday_order)

heatmap_data

departure_hour,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,...,14.0,15.0,16.0,17.0,18.0,19.0,20.0,21.0,22.0,23.0
weekday,,,,,,,,,,,,,,,,,,,,,
Monday,3.910176,3.500577,1.696456,1.503336,1.880946,2.290649,2.407457,2.081602,1.922777,1.770067,...,2.300110,2.558335,2.213203,2.269628,2.156004,1.927911,1.680505,1.390417,1.603820,1.933059
Tuesday,1.539925,1.707508,1.576268,1.448706,2.006419,2.388072,2.729272,2.519010,2.172325,2.049639,...,2.476297,2.753405,2.478564,2.253075,1.974499,1.968334,1.761403,1.522724,1.484125,1.819303
Wednesday,1.259009,2.376003,1.617043,1.320078,1.795172,2.214976,2.341993,2.216309,1.888476,1.827815,...,2.545051,2.828286,2.454018,2.284689,2.240449,2.084859,1.764041,1.443534,1.659191,1.775772
Thursday,2.585386,2.011318,1.573670,1.556642,1.954567,2.369473,2.625829,2.404493,2.100807,1.872569,...,2.322225,2.732033,2.485415,2.316966,2.064068,1.990623,1.660735,1.403687,1.358436,1.442208
Friday,1.144629,1.570074,1.418238,1.463994,1.930933,2.268178,2.414866,2.313128,2.113925,2.074963,...,2.481975,2.647602,2.453096,2.366320,2.304369,2.248739,1.770452,1.564820,1.332298,1.528809
Saturday,0.449254,0.571426,1.050694,0.869034,1.019356,1.114148,1.381388,1.500098,1.386659,1.565741,...,1.608534,1.659343,1.524964,1.530757,1.374236,1.434721,1.174938,1.033972,0.976668,1.020234
Sunday,0.553174,1.054768,1.288297,0.803299,0.967207,1.026761,1.083983,1.287959,1.075973,1.233318,...,1.654569,1.802138,1.553772,1.766442,1.445288,1.296676,1.320353,1.218527,1.151065,1.440234


In [29]:
fig = px.imshow(
    heatmap_data,
    aspect="auto",
    title="Average Delay by Weekday and Hour",
    labels={
        "x": "Hour of Day",
        "y": "Weekday",
        "color": "Delay (min)"
    }
)

fig.show()

In [30]:
top_station_pairs.head()

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips
0,PA,MLX,38.968315,26.083333,91
1,MLX,ND G,33.543229,14.991667,32
2,MKI,I W1,23.970000,24.733333,45
3,BPA,VBO,22.715568,15.433333,455
4,PAG,MLX,21.901109,11.666667,1307


## 7. Most delayed station pairs

This chart shows the station pairs with the highest average delay.

To avoid unreliable results, only station pairs with at least 30 observations were included.

In [31]:
top_station_pairs_plot = (
    top_station_pairs
    .head(15)
    .copy()
)

top_station_pairs_plot["station_pair"] = (
    top_station_pairs_plot["start_betriebsstelle"]
    + " → "
    + top_station_pairs_plot["ziel_betriebsstelle"]
)

top_station_pairs_plot.head()

,start_betriebsstelle,ziel_betriebsstelle,mean_delay,median_delay,number_of_trips,station_pair
0,PA,MLX,38.968315,26.083333,91,PA → MLX
1,MLX,ND G,33.543229,14.991667,32,MLX → ND G
2,MKI,I W1,23.970000,24.733333,45,MKI → I W1
3,BPA,VBO,22.715568,15.433333,455,BPA → VBO
4,PAG,MLX,21.901109,11.666667,1307,PAG → MLX


In [32]:
fig = px.bar(
    top_station_pairs_plot.sort_values("mean_delay"),
    x="mean_delay",
    y="station_pair",
    orientation="h",
    title="Top 15 Most Delayed Station Pairs"
)

fig.show()

### Note

The current visualization uses operational station codes from the MMTIS dataset.

In a later step, these codes will be mapped to human-readable station names using GTFS station information.